# 06 — End-to-End QEC Research: Threshold Curves & Resource Estimation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/06_qec_research.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/06_qec_research.ipynb)


This notebook demonstrates QuantX's fault-tolerant quantum computing (FTQC) pipeline,
including:

1. **Surface-code threshold curves** for d = 3, 5, 7 under the code-capacity noise model
2. **Distance vs. logical error rate** at a fixed physical error rate
3. **FTQC resource estimation** via the Fowler 2012 formula (M156)
4. **Bosonic cat-qubit comparison** — how cat qubits reduce overhead relative to surface codes

### References
- Fowler, A. G. et al. (2012). *Phys. Rev. A* 86, 032324. https://doi.org/10.1103/PhysRevA.86.032324
- Dennis, E. et al. (2002). *J. Math. Phys.* 43, 4452. https://doi.org/10.1063/1.1499754
- Google AI. (2023). *Nature* 614, 676–681. https://doi.org/10.1038/s41586-022-05434-1

In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from quantx_engine.contrib.ftqc_pipeline import (
    FTQCPipeline,
    threshold_curve,
    threshold_crossing,
    FTQCResourceEstimator,
    ResourceEstimate,
)

print('QuantX imports OK')

## 1. Surface-Code Threshold Curve

We sweep the physical error rate p ∈ [3%, 15%] for distances d = 3, 5, 7.
The threshold p_th is the crossing point: below p_th, increasing d strictly
reduces the logical error rate (exponentially).

> **Published benchmark**: Dennis et al. (2002) report p_th ≈ 10.3% for the
> toric code under the code-capacity model.  QuantX's `MWPMDecoder` matches
> exactly up to 10 defects and greedily beyond that, which the larger codes
> reach near threshold, so expect a somewhat lower estimate here.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
# Sized to finish in under a minute.  The decoder is pure Python and the largest
# distance dominates the cost.  For publication-quality curves add 9 to DISTANCES
# and use 5000+ shots; expect this cell to take tens of minutes.
DISTANCES  = [3, 5, 7]
N_SHOTS    = 600
SEED       = 42

print(f'Running threshold curve: distances={DISTANCES}, n_shots={N_SHOTS} ...')
curve = threshold_curve(
    distances=DISTANCES,
    p_phys_range=(0.03, 0.15),
    n_points=10,
    n_shots=N_SHOTS,
    n_rounds=1,        # code-capacity model
    decoder='mwpm',
    seed=SEED,
)

th = threshold_crossing(curve)
print(f'Estimated threshold: p_th ≈ {th:.4f}  ({th*100:.2f}%)')
print(f'Published benchmark (Dennis 2002, code-capacity): ~10.3%')

In [ ]:
# ── Plot threshold curves ─────────────────────────────────────────────────────
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
fig, ax = plt.subplots(figsize=(7, 5))

for d, color in zip(DISTANCES, colors):
    ax.plot(np.array(curve.p_phys) * 100,
            np.array(curve.p_logical[d]) * 100,
            'o-', color=color, label=f'd = {d}', linewidth=2, markersize=5)

if th is not None:
    ax.axvline(th * 100, color='k', linestyle='--', linewidth=1.2,
               label=f'p_th ≈ {th*100:.1f}%')

ax.set_xlabel('Physical error rate p (%)', fontsize=12)
ax.set_ylabel('Logical error rate p_L (%)', fontsize=12)
ax.set_title('Surface-code threshold curves\n(code-capacity model, MWPM decoder)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('qec_threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → qec_threshold_curves.png')

## 2. Distance vs. Logical Error Rate (fixed p_phys)

Below threshold, p_L scales as ~ (p/p_th)^⌊(d+1)/2⌋.  We plot this
relationship for a fixed physical error rate p = 0.1% — typical for
superconducting qubits around 2023.

In [ ]:
P_PHYS    = 0.001   # 0.1%
DIST_SCAN = list(range(3, 22, 2))  # d = 3, 5, 7, ..., 21

p_L_values = []
for d in DIST_SCAN:
    pipe = FTQCPipeline(code_distance=d, p_physical=P_PHYS, decoder='mwpm')
    pL   = pipe.logical_error_rate(n_shots=2000, n_rounds=1, seed=SEED + d)
    p_L_values.append(pL)
    print(f'd={d:2d}  p_L={pL:.4e}')

In [ ]:
# Fowler 2012 analytic curve for comparison
def fowler_pL(d, p_phys, p_th=0.01, A=0.1):
    t = (d + 1) // 2
    return A * (p_phys / p_th) ** t

d_fine   = np.linspace(3, 21, 200)
pL_fine  = [fowler_pL(d, P_PHYS) for d in d_fine]

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(DIST_SCAN, p_L_values, 'o', color='#1f77b4',
            markersize=7, label='QuantX Monte Carlo')
ax.semilogy(d_fine, pL_fine, '--', color='#ff7f0e', linewidth=2,
            label='Fowler 2012 analytic')
ax.set_xlabel('Code distance d', fontsize=12)
ax.set_ylabel('Logical error rate p_L', fontsize=12)
ax.set_title(f'p_L vs. code distance  (p_phys = {P_PHYS*100:.1f}%)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.savefig('qec_distance_vs_logical.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → qec_distance_vs_logical.png')

## 3. FTQC Resource Estimation (Fowler 2012)

Given a **target logical error rate** (e.g. 10⁻¹⁰ for Shor's algorithm on RSA-2048)
and a **physical error rate**, `FTQCResourceEstimator` finds the minimum code
distance and estimates physical-qubit overhead.

Formula:
```
p_L ≈ A × (p_phys / p_th)^⌊(d+1)/2⌋,   A = 0.1,  p_th = 0.01
physical_qubits = 2d²
```

In [ ]:
# ── Sweep physical error rate → physical qubit overhead ───────────────────────
TARGET_PL  = 1e-10   # fault-tolerant Shor's algorithm budget per gate
p_scan     = np.logspace(-4, -2, 25)   # 0.01% to 1%

results = []
for p in p_scan:
    try:
        est = FTQCResourceEstimator(
            target_logical_error_rate=TARGET_PL,
            physical_error_rate=p,
        )
        r = est.estimate()
        results.append((p, r))
    except Exception:
        pass

p_vals   = [x[0] for x in results]
d_vals   = [x[1].code_distance for x in results]
qb_vals  = [x[1].physical_qubits for x in results]
pL_vals  = [x[1].logical_error_rate for x in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.semilogx(np.array(p_vals) * 100, d_vals, 'o-', color='#2ca02c', linewidth=2)
ax1.set_xlabel('Physical error rate (%)', fontsize=12)
ax1.set_ylabel('Minimum code distance d', fontsize=12)
ax1.set_title(f'Code distance for p_L < {TARGET_PL:.0e}', fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.semilogx(np.array(p_vals) * 100, qb_vals, 's-', color='#d62728', linewidth=2)
ax2.set_xlabel('Physical error rate (%)', fontsize=12)
ax2.set_ylabel('Physical qubits (2d²)', fontsize=12)
ax2.set_title(f'Physical qubit overhead for p_L < {TARGET_PL:.0e}', fontsize=11)
ax2.grid(True, alpha=0.3)

fig.suptitle('FTQC Resource Estimation via Fowler 2012', fontsize=13)
fig.tight_layout()
plt.savefig('qec_resource_estimation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → qec_resource_estimation.png')

In [ ]:
# ── Concrete example: Shor's algorithm on RSA-2048 ────────────────────────────
for p_phys, label in [(1e-3, '0.1% (near-term SC)'), (1e-4, '0.01% (future SC)')]:
    est = FTQCResourceEstimator(
        target_logical_error_rate=1e-10,
        physical_error_rate=p_phys,
    )
    r = est.estimate()
    print(f'p_phys = {label}')
    print(f'  Code distance        : d = {r.code_distance}')
    print(f'  Physical qubits      : {r.physical_qubits}')
    print(f'  Logical error rate   : {r.logical_error_rate:.2e}')
    print(f'  Magic-state levels   : {r.magic_states_per_t} (T-gate distillation)')
    print(f'  Extra qubits for T   : {r.physical_qubits_for_t}')
    print(f'  QEC cycles / gate    : {r.cycles}')
    print()

## 4. Bosonic Cat-Qubit Comparison

Cat qubits exploit the exponential suppression of bit-flip errors in oscillator
Hilbert space.  For mean photon number $\bar{n}$, the effective bit-flip rate
scales as $p_{\text{flip}} \sim e^{-2\bar{n}}$, while phase-flip errors grow only
polynomially.

We model this with the rates of the QuantX `CatNoiseModel`,
then compare the required code distance against an equal-rate surface code.

Reference: Mirrahimi, M. et al. (2014). *New J. Phys.* 16, 045014.
https://doi.org/10.1088/1367-2630/16/4/045014

In [ ]:
from quantx_engine.contrib.bosonic import CatNoiseModel

# Mean photon numbers to scan
nbar_range = np.linspace(1, 10, 20)

cat_bit_flip  = []
cat_phase_flip = []
for nbar in nbar_range:
    # n̄ ≈ |α|² at these cat sizes.
    noise = CatNoiseModel(kappa_1=1e-3, kappa_phi=1e-3, alpha_sq=float(nbar))
    cat_bit_flip.append(noise.bit_flip_rate)
    cat_phase_flip.append(noise.phase_flip_rate)

# Surface-code effective p_phys = symmetric depolarizing ≈ (bit_flip + phase_flip)/2
sc_effective = [(b + ph) / 2 for b, ph in zip(cat_bit_flip, cat_phase_flip)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: error rates vs mean photon number
ax = axes[0]
ax.semilogy(nbar_range, cat_bit_flip, 'b-o', markersize=5, label='Bit-flip (cat)')
ax.semilogy(nbar_range, cat_phase_flip, 'r-s', markersize=5, label='Phase-flip (cat)')
ax.semilogy(nbar_range, sc_effective, 'g--', linewidth=2, label='SC effective p_phys')
ax.set_xlabel(r'Mean photon number $\bar{n}$', fontsize=12)
ax.set_ylabel('Error rate', fontsize=12)
ax.set_title('Cat-qubit noise vs. photon number', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# Right: required code distance vs mean photon number
ax = axes[1]
cat_distances = []
sc_distances  = []
for nbar, bf, pf, sc_p in zip(nbar_range, cat_bit_flip, cat_phase_flip, sc_effective):
    # For cat: use max(bit_flip, phase_flip) as effective physical rate
    p_cat = max(bf, pf)
    for p_eff, store in [(p_cat, cat_distances), (sc_p, sc_distances)]:
        if 0 < p_eff < 0.01:  # below threshold
            try:
                est  = FTQCResourceEstimator(
                    target_logical_error_rate=1e-6,
                    physical_error_rate=p_eff,
                )
                store.append(est.estimate().code_distance)
            except Exception:
                store.append(np.nan)
        else:
            store.append(np.nan)

ax.plot(nbar_range, cat_distances, 'b-o', markersize=5, label='Cat qubit (max error)')
ax.plot(nbar_range, sc_distances, 'g--s', markersize=5, label='Surface code (symmetric)')
ax.set_xlabel(r'Mean photon number $\bar{n}$', fontsize=12)
ax.set_ylabel('Required code distance d', fontsize=12)
ax.set_title(r'Code distance for p_L < $10^{-6}$', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle('Cat-qubit vs. surface-code overhead comparison', fontsize=13)
fig.tight_layout()
plt.savefig('qec_cat_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → qec_cat_comparison.png')

## Summary

| Result | Value |
|--------|-------|
| Estimated threshold (code-capacity, d = 3–7, 600 shots) | ~8–10% |
| Published (Dennis 2002) | 10.3% |
| Physical qubits @ p=0.1%, d satisfying p_L < 1e-10 | ~722 (d = 19) |
| Physical qubits @ p=0.01%, p_L < 1e-10 | ~162 (d = 9) |

With $\kappa_1 = \kappa_\phi = 10^{-3}$ the cat's phase-flip rate grows linearly with
$\bar{n}$ and dominates. Fed to the same surface-code estimator as
max(bit-flip, phase-flip), the cat needs a *larger* distance than the symmetric
model at every $\bar{n}$ shown, and above $\bar{n} \approx 5$ its phase-flip rate
passes the 1% threshold. The bias pays off only with an outer code built for it,
such as the repetition code against phase flips in notebook 07; this estimator
does not model one.